# Zerobus Ingest — One Click Demo

Clone this notebook, then click Run All

## Auto configure [Public Demo](https://docs.databricks.com/aws/en/ingestion/zerobus-ingest)

- ZeroBus endpoint and workspace URL — [Get your workspace URL and ZeroBus Ingest endpoint](https://docs.databricks.com/aws/en/ingestion/zerobus-ingest#get-your-workspace-url-and-zerobus-ingest-endpoint)
- Service principal + OAuth client secret — run **`notebooks/zerobus_service_principal.ipynb`** once (same Databricks secret as below), then continue here. [Create a service principal and grant permissions](https://docs.databricks.com/aws/en/ingestion/zerobus-ingest#create-a-service-principal-and-grant-permissions). That notebook sets SP **display name** to `zerobus-sp--<scope>--<secretKey>--<jsonField>` matching where `ZEROBUS_OAUTH_SECRET` is stored.
- OATH SECRET - [OAuth secret](https://docs.databricks.com/aws/en/dev-tools/auth/oauth-m2m)
- Grant use, select, modify permission on Catalog, Schema, Table

## Optional: OpenTelemetry + Grafana

For traces/metrics to Grafana Cloud with ZeroBus, use **`notebooks/otel/grafana/zerobus-otel.ipynb`**. Configure the Databricks secret first with **`notebooks/otel/grafana/secrets-bootstrap.ipynb`** (Grafana Connections + OTLP fields are documented there).

## Normal table ingestion

Create 10 rows of the following
```
    for i in range(10):
        record_dict = {
            "device_name": f"sensor-{i}",
            "temp": 20 + i % 15,
            "humidity": 50 + i % 40
        }
        last_offset = stream.ingest_record_offset(record_dict)
```

In [ ]:
%pip install --quiet databricks-zerobus-ingest-sdk

In [ ]:
# config for reuse 

_config = {
  "ZEROBUS_TABLE_NAME": "air_quality",
  "ZEORBUS_SCHEMA": "",                 # will be your name if not populated
  "ZEORBUS_CATALOG": "",                # will be main if not populated

  # save if generated by this notebook
  "ZEROBUS_SERVICE_PRINCIPAL_NAME": "", # set by notebooks/zerobus_service_principal.ipynb (Databricks secret)
  "ZEROBUS_SERVICE_PRINCIPAL_ID": "",   # set by notebooks/zerobus_service_principal.ipynb
  "ZEROBUS_APP_ID": "",                 # set by notebooks/zerobus_service_principal.ipynb
  "ZEROBUS_OAUTH_SECRET": "",           # set by notebooks/zerobus_service_principal.ipynb (Databricks secret)

  # if this notebook cannot figure it out
  "DATABRICKS_WORKSPACE_URL": "", # auto detect if not populated or incorrect
  "ZEROBUS_SERVER_ENDPOINT": "",  # auto detect if not populated or incorrect
}

# import zerobus 

from databricks.sdk import WorkspaceClient
from zerobus.sdk.sync import ZerobusSdk
from zerobus.sdk.shared import RecordType, StreamConfigurationOptions, TableProperties, HeadersProvider
import re, time
_w = WorkspaceClient()
username = re.sub(r"[^a-z0-9]", "_", _w.current_user.me().user_name.split("@")[0])

# merge to _config saved config.json from the file
import json
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / ".git").exists() or (parent / "config.json").exists():
            return parent
    return start

_SECRET_SCOPE = "lfczerobusdemo"
_SECRET_KEY   = "lfczerobusdemo"

def _ensure_secret_scope_and_key(w) -> None:
    """Create the secret scope and an empty key if they do not exist yet.

    Raises on any error other than scope/key already existing.
    """
    from databricks.sdk.errors import ResourceAlreadyExists, ResourceDoesNotExist
    try:
        w.secrets.create_scope(_SECRET_SCOPE)
        print(f"Created secret scope {_SECRET_SCOPE!r}")
    except ResourceAlreadyExists:
        pass

    try:
        w.secrets.get_secret(_SECRET_SCOPE, _SECRET_KEY)
    except ResourceDoesNotExist:
        w.secrets.put_secret(scope=_SECRET_SCOPE, key=_SECRET_KEY, string_value="{}")
        print(f"Created empty secret {_SECRET_SCOPE!r}/{_SECRET_KEY!r}")

def _load_saved(w) -> dict:
    """Load saved config from Databricks secret.

    Creates the scope and key (with empty content) on first use if not found.
    Raises on any other error (permissions, network, etc.).

    dbutils is available in both Databricks workspace notebooks and locally via
    Databricks Connect (databricks.sdk.dbutils.RemoteDbUtils).
    """
    try:
        raw = dbutils.secrets.get(scope=_SECRET_SCOPE, key=_SECRET_KEY)
        print(f"Loaded config from secret scope={_SECRET_SCOPE!r} key={_SECRET_KEY!r}")
        return json.loads(raw)
    except Exception:
        pass  # scope or key not found — create them below

    _ensure_secret_scope_and_key(w)
    return {}

def _load_config(defaults: dict, w) -> tuple[dict, dict]:
    """Merge saved config into defaults.

    Returns (merged, saved_snapshot).
    Notebook _config wins when non-blank; saved values fill in blank defaults.
    """
    saved = _load_saved(w)
    merged = {k: v if v else saved.get(k, v) for k, v in defaults.items()} | {
        k: v for k, v in saved.items() if k not in defaults
    }
    return merged, dict(saved)

def _save_config(updates: dict, w) -> None:
    """Persist updates to the Databricks secret. Raises on failure."""
    from databricks.sdk.errors import ResourceDoesNotExist
    _client = w
    try:
        current = json.loads(dbutils.secrets.get(scope=_SECRET_SCOPE, key=_SECRET_KEY))
    except ResourceDoesNotExist:
        current = {}
    current.update(updates)
    _client.secrets.put_secret(scope=_SECRET_SCOPE, key=_SECRET_KEY, string_value=json.dumps(current, indent=2))
    print(f"Saved {list(updates.keys())} to secret {_SECRET_SCOPE}/{_SECRET_KEY}")

def _save_config_if_changed(current: dict, original: dict, w) -> None:
    """Save only the keys that changed relative to the loaded snapshot."""
    changed = {k: v for k, v in current.items() if original.get(k) != v}
    if changed:
        _save_config(changed, w)
    else:
        print("Config unchanged — nothing to save")

_repo_root = _find_repo_root(Path(__file__).parent if "__file__" in dir() else Path.cwd())
_config, _config_original = _load_config(_config, _w)

In [ ]:
# auto determine SERVER_ENDPOINT, DATABRICKS_WORKSPACE_URL

def get_endpoint_workspace_url(SERVER_ENDPOINT, DATABRICKS_WORKSPACE_URL):
    generated=False
    _url_mismatch = DATABRICKS_WORKSPACE_URL and DATABRICKS_WORKSPACE_URL != _w.config.host
    if _url_mismatch:
        print(f"Warning: DATABRICKS_WORKSPACE_URL={DATABRICKS_WORKSPACE_URL!r} does not match "
            f"WorkspaceClient host={_w.config.host!r}. Recalculating both values.")
        SERVER_ENDPOINT = ""
        DATABRICKS_WORKSPACE_URL = ""

    if not SERVER_ENDPOINT or not DATABRICKS_WORKSPACE_URL:
        if not DATABRICKS_WORKSPACE_URL:
            DATABRICKS_WORKSPACE_URL = _w.config.host
        if not SERVER_ENDPOINT:
            _workspace_id = _w.get_workspace_id()
            if "azuredatabricks.net" in DATABRICKS_WORKSPACE_URL:
                _region = spark.sql("SELECT current_metastore()").collect()[0][0].split(":")[1]
            else:
                _region = _w.clusters.list_zones().default_zone[:-1]     # AWS: strip trailing AZ letter
            _domain = "azuredatabricks.net" if "azuredatabricks.net" in DATABRICKS_WORKSPACE_URL else "cloud.databricks.com"
            SERVER_ENDPOINT = f"https://{_workspace_id}.zerobus.{_region}.{_domain}"
        generated=True
        print(f"Auto-derived SERVER_ENDPOINT={SERVER_ENDPOINT}")
        print(f"Auto-derived DATABRICKS_WORKSPACE_URL={DATABRICKS_WORKSPACE_URL}")

    return SERVER_ENDPOINT, DATABRICKS_WORKSPACE_URL, generated

_config["ZEROBUS_SERVER_ENDPOINT"], _config["DATABRICKS_WORKSPACE_URL"], generated = get_endpoint_workspace_url(
    _config.get("ZEROBUS_SERVER_ENDPOINT",""), 
    _config.get("DATABRICKS_WORKSPACE_URL",""))
print(f"{_config['ZEROBUS_SERVER_ENDPOINT']=}")
print(f"{_config['DATABRICKS_WORKSPACE_URL']=}")

In [ ]:
# ZeroBus bootstrap lives in **`notebooks/zerobus_service_principal.ipynb`** (run once; writes SP ids + OAuth client secret to this notebook's Databricks secret).
# Re-merge from the secret in case you ran that notebook after starting this kernel.

_sp_keys = (
    "ZEROBUS_SERVICE_PRINCIPAL_NAME",
    "ZEROBUS_SERVICE_PRINCIPAL_ID",
    "ZEROBUS_APP_ID",
    "ZEROBUS_OAUTH_SECRET",
)
try:
    _saved_sp = json.loads(dbutils.secrets.get(scope=_SECRET_SCOPE, key=_SECRET_KEY))
except Exception:
    _saved_sp = {}
for _k in _sp_keys:
    if not str(_config.get(_k, "")).strip() and str(_saved_sp.get(_k, "")).strip():
        _config[_k] = _saved_sp[_k]

if (
    not str(_config.get("ZEROBUS_SERVICE_PRINCIPAL_ID", "")).strip()
    or not str(_config.get("ZEROBUS_APP_ID", "")).strip()
    or not str(_config.get("ZEROBUS_OAUTH_SECRET", "")).strip()
):
    raise RuntimeError(
        "Missing ZEROBUS_SERVICE_PRINCIPAL_ID, ZEROBUS_APP_ID, or ZEROBUS_OAUTH_SECRET. "
        "Run notebooks/zerobus_service_principal.ipynb on this workspace, then re-run this notebook from the config cell (or Run All)."
    )

print(f"ZEROBUS_SERVICE_PRINCIPAL_ID = {_config['ZEROBUS_SERVICE_PRINCIPAL_ID']}")
print(f"ZEROBUS_APP_ID               = {_config['ZEROBUS_APP_ID']}")
print("ZEROBUS_OAUTH_SECRET         = <set>")


In [ ]:
# OAuth client secret is minted in **`notebooks/zerobus_service_principal.ipynb`**. Sanity-check OIDC here.

def _credentials_valid(client_id: str, client_secret: str, workspace_url: str) -> bool:
    if not client_id or not client_secret:
        return False
    try:
        import requests as _req

        _resp = _req.post(
            f"{workspace_url.rstrip('/')}/oidc/v1/token",
            data={
                "grant_type": "client_credentials",
                "client_id": client_id,
                "client_secret": client_secret,
                "scope": "all-apis",
            },
            timeout=10,
        )
        if not _resp.ok:
            print(f"OIDC check: {_resp.status_code} {_resp.text}")
        return _resp.ok
    except Exception as ex:
        print(f"OIDC check: {ex}")
        return False


if not _credentials_valid(
    _config["ZEROBUS_APP_ID"],
    _config["ZEROBUS_OAUTH_SECRET"],
    _config["DATABRICKS_WORKSPACE_URL"],
):
    raise RuntimeError(
        "OAuth client id/secret failed OIDC client_credentials check. "
        "Re-run notebooks/zerobus_service_principal.ipynb (it remints the secret when invalid), then re-run this notebook from the config cell."
    )
print("OAuth client credentials OK (OIDC).")

In [ ]:
# create target schema, table

fq_table_name_parts = _config['ZEROBUS_TABLE_NAME'].split('.')
table_catalog = "main" if len(fq_table_name_parts) <= 2 else fq_table_name_parts.pop(0)
table_schema = username if len(fq_table_name_parts) < 2 else fq_table_name_parts.pop(0)
table_name = fq_table_name_parts.pop(0) if len(fq_table_name_parts) > 0 else "air_quality"
fq_table_name=f"{table_catalog}.{table_schema}.{table_name}"

if not table_name: raise Exception("table not specfied")

print(f"{table_catalog=} {table_schema=} {table_name=}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {table_catalog}.{table_schema}")

# Drop and recreate if the schema has been corrupted by a previous Scenario 2 run.
_expected = {"device_name", "temp", "humidity"}
try:
    _actual = {f.name for f in spark.table(fq_table_name).schema}
    if not _expected.issubset(_actual):
        print(f"Schema mismatch (found {_actual}), dropping and recreating...")
        spark.sql(f"DROP TABLE IF EXISTS {fq_table_name}")
except Exception:
    pass  # table doesn't exist yet — CREATE below will handle it

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {fq_table_name} (
  device_name STRING,
  temp        INT,
  humidity    INT
)
""")


In [ ]:
# grant use on catalog, schema, select modify on target table

def _has_grant(principal, action, securable_type, securable_name):
    """Return True if `principal` already has `action` on the given securable."""
    rows = spark.sql(f"SHOW GRANTS `{principal}` ON {securable_type} {securable_name}").collect()
    return any(r["ActionType"] == action for r in rows)

# Only grant if the principal doesn't already have the privilege
if not _has_grant(_config['ZEROBUS_APP_ID'], "USE CATALOG", "CATALOG", table_catalog):
    spark.sql(f"GRANT USE CATALOG ON CATALOG {table_catalog} TO `{_config['ZEROBUS_APP_ID']}`")
    print(f"WARNING: Granted USE CATALOG on {table_catalog}")
else:
    print(f"USE CATALOG on {table_catalog} already granted")

if not _has_grant(_config['ZEROBUS_APP_ID'], "USE SCHEMA", "SCHEMA", f"{table_catalog}.{table_schema}"):
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA {table_catalog}.{table_schema} TO `{_config['ZEROBUS_APP_ID']}`")
    print(f"WARNING: Granted USE SCHEMA on {table_catalog}.{table_schema}")
else:
    print(f"USE SCHEMA on {table_catalog}.{table_schema} already granted")

for action in ["MODIFY", "SELECT"]:
    if not _has_grant(_config['ZEROBUS_APP_ID'], action, "TABLE", fq_table_name):
        spark.sql(f"GRANT {action} ON TABLE {fq_table_name} TO `{_config['ZEROBUS_APP_ID']}`")
        print(f"WARNING: Granted {action} on {fq_table_name}")
    else:
        print(f"{action} on {fq_table_name} already granted")

In [ ]:
# url of the table, table storage, and reject rows

_table_url = f"{_config['DATABRICKS_WORKSPACE_URL'].rstrip('/')}/explore/data/{table_catalog}/{table_schema}/{table_name}"
print(f"Table: {_table_url}")

_detail = spark.sql(f"DESCRIBE DETAIL {fq_table_name}").collect()[0]
_location = _detail["location"]
_rejected_path = f"{_location}/_zerobus/table_rejected_parquets/"

print(f"Table storage : {_location}")
print(f"Rejected rows : {_rejected_path}")

In [ ]:
# Normal ingest — 10 rows of valid data matching the table schema

sdk = ZerobusSdk(
    _config['ZEROBUS_SERVER_ENDPOINT'],
    _config['DATABRICKS_WORKSPACE_URL']
)

table_properties = TableProperties(fq_table_name)
options = StreamConfigurationOptions(record_type=RecordType.JSON)
stream = sdk.create_stream(_config['ZEROBUS_APP_ID'], _config['ZEROBUS_OAUTH_SECRET'], table_properties, options)

try:
    last_offset = None
    for i in range(10):
        record_dict = {
            "device_name": f"sensor-{i}",
            "temp": 20 + i % 15,
            "humidity": 50 + i % 40
        }
        last_offset = stream.ingest_record_offset(record_dict)

    # Wait once for the final offset — all prior offsets are guaranteed committed too
    if last_offset is not None:
        stream.wait_for_offset(last_offset)
finally:
    stream.close()


In [ ]:
# min max on device_name, count

display(
spark.sql(f"select min(device_name), max(device_name), count(*) from {fq_table_name}")
)


In [ ]:
# row count by device id

display(
spark.sql(f"""
SELECT device_name, count(*) AS row_count
FROM {fq_table_name}
GROUP BY device_name
HAVING count(*) > 1
ORDER BY CAST(regexp_extract(device_name, '(\\\\d+)$', 1) AS INT), row_count DESC
""")
)

In [ ]:
# show num of records processed each time we called zeorbus 
# there will be some delays

try:
    display(spark.sql(f"""
    SELECT commit_time, table_name, committed_records, errors
    FROM system.lakeflow.zerobus_ingest
    WHERE table_name = '{fq_table_name}'
    AND workspace_id = '{str(_w.get_workspace_id())}'
    ORDER BY commit_time DESC
    LIMIT 20
    """))
except Exception as e:
    if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
        print("system.lakeflow.zerobus_ingest is not available in this workspace.")
        print("Ask a workspace admin to enable the ZeroBus Ingest system table.")
    else:
        raise


In [ ]:
# save config if anything changed

_save_config_if_changed(_config, _config_original, _w)